<div style="text-align: center"><img src="https://telematica.usm.cl/wp-content/themes/telematica-web/assets/img/header-usm.png" height="75"></div>

# Tarea 3: uIP y RPL
### Asignatura: Redes de Sensores - TEL329 - 2026
#### Docente y autor: Berioska Contreras Vargas - berioska.contreras@usm.cl

###### Este contenido es parte de la asignatura TEL329 de UTFSM. Las tareas son un complemento a las lecciones teóricas. El objetivo es simular y emular redes de sensores constreñidos.

## Introducción a uIP

Contiki soporta direccionamiento IP a través del apilamiento *uIP* TCP/IP. Contiki también tiene soporte para enrutamiento IP utilizando el protocolo RPL que ha sido estandarizado por el IETF-RFC-6550. Las redes RPL están ancladas en una raíz que típicamente actúa como enrutador de borde 6LowPAN, p.e. provee una malla de red de baja-potencia con conexión a redes externas (Internet).

Utilizaremos un ejemplo de IPv6 ubicado en la ruta $\$(CONTIKI)/examples/rpl-border-router$. Este ejemplo ilustra como crear una red IPv6 simple en el simulador Cooja. Utilizaremos el *Mote* *Cooja* que actuará como enrutador de borde RPL para nuestra red IPv6. El enrutador de borde configurará el prefijo IPv6 de la red e iniciará la creación del árbol de enrutamiento RPL. Para crear un nodo enrutador utilizaremos el código disponible en $\$(CONTIKI)/examples/rpl-border-router/border-router.c$. Una vez incluido el enrutador de borde, es necesario abrir un puerto al presionar click derecho en el nodo, luego seleccionar *"Mote tools for Contiki > Seial Socket (Server)*. Una ventana emergente notificará que el enrutador de borde se encuentra escuchando en el puerto 60001. 

Complementariamente, crearemos un tunel de reenvío de paquetes desde los nodos hacia los destino (redes externas) a través del enrutador de borde siguiendo el Paso 1.

| Fuente | Referencia | 
| - | - |
| [Contiki-NG Documentation](https://docs.contiki-ng.org/en/develop/) | Documentación de Contiki-NG |


### Paso 1 - Crear el túnel

```bash
#Paso 1 - Creación de túnel
$ cd $(CONTIKI)/tools/serial-io/
$ make tunslip6
$ sudo su
$ ./tunslip6 -a 127.0.0.1 fd00::1/64

#salida esperada
slip connected to ''127.0.0.1:60001''
opened tun device ''/dev/tun0''
ifconfig tun0 inet 'hostname' up
ifconfig tun0 add fd00::1/64/64
ifconfig tun0 add fe80::0:0:0:1/64
ifconfig tun0
tun0 Link encap:UNSPEC HWaddr 00-00-00-00-00-00-00-00-00-00-00-00-00-00-00-00
inet addr:127.0.1.1 P-t-P:127.0.1.1 Mask:255.255.255.255
inet6 addr: fe80::1/64 Scope:Link
inet6 addr: fd00::1/64/64 Scope:Global
UP POINTOPOINT RUNNING NOARP MULTICAST MTU:1500 Metric:1
RX packets:0 errors:0 dropped:0 overruns:0 frame:0
TX packets:0 errors:0 dropped:0 overruns:0 carrier:0
collisions:0 txqueuelen:500
RX bytes:0 (0.0 B) TX bytes:0 (0.0 B)
```


* Comprobaremos la disponibilidad del túnel IPv6 utilizando el comando *ifconfig* en otra terminal. Y validaremos la accesibilidad al enrutador de borde utilizando el comando $ping$ junto a su dirección en el terminal (utilizar el comando ping6). Para ver el listado del vecindario del enrutador y sus rutas, escriba la dirección IP confinada en paréntesis cuadrados en el navegador. Ver Paso 2.

| Protocolo | Descripción | 
| - | - |
| [RPL Border Router](https://docs.contiki-ng.org/en/develop/doc/tutorials/RPL-border-router.html) | RPL Border de Contiki-NG |
| [RPL Deep Dive](https://docs.contiki-ng.org/en/develop/doc/tutorials/RPL.html#deep-dive)| RPL Deep Dive |

### Paso 2 - Verificar túnel y conectividad

```bash
#Paso 2 - verificación tunel
$ ifconfig tun0

#Paso 2 - ping6 hacia enrutador de borde
$ ping6 <server ipv6 address>
```


## Implementación UDP Server/Client

Múltiples clientes UDP se unen a la red DODAG y envían peticiones periódicas al nodo raíz. Examine la ejecución de una red RPL en Cooja utilizando el código $examples/rpl-udp$.

| Archivo |	RPL role |	UDP role |
| -- | -- |-- |
| udp-server.c |	NETSTACK_ROUTING.root_start() |	listen 5678, echo payload |
| udp-client.c |	join DODAG, wait until reachable |	send to root every ~10 s on 8765 |

### Código base: udp-server.c

```c
#include "contiki.h"
#include "net/routing/routing.h"
#include "net/netstack.h"
#include "net/ipv6/simple-udp.h"
#include "sys/log.h"

#define LOG_MODULE "App"
#define LOG_LEVEL LOG_LEVEL_INFO

#define WITH_SERVER_REPLY  1
#define UDP_CLIENT_PORT    8765
#define UDP_SERVER_PORT    5678

static struct simple_udp_connection udp_conn;

PROCESS(udp_server_process, "UDP server");
AUTOSTART_PROCESSES(&udp_server_process);

static void
udp_rx_callback(struct simple_udp_connection *c,
                const uip_ipaddr_t *sender_addr,
                uint16_t sender_port,
                const uip_ipaddr_t *receiver_addr,
                uint16_t receiver_port,
                const uint8_t *data,
                uint16_t datalen)
{
  LOG_INFO("Received request '%.*s' from ", datalen, (char *)data);
  LOG_INFO_6ADDR(sender_addr);
  LOG_INFO_("\n");

#if WITH_SERVER_REPLY
  LOG_INFO("Sending response.\n");
  simple_udp_sendto(&udp_conn, data, datalen, sender_addr);
#endif
}

PROCESS_THREAD(udp_server_process, ev, data)
{
  PROCESS_BEGIN();

  /* Become DODAG root: advertise prefix, accept DAOs, enable downward routes */
  NETSTACK_ROUTING.root_start();

  /* Bind local 5678, accept datagrams from any client port 8765 */
  simple_udp_register(&udp_conn, UDP_SERVER_PORT, NULL,
                      UDP_CLIENT_PORT, udp_rx_callback);

  PROCESS_END();
}
```


### Código base: udp-client.c

```c
#include "contiki.h"
#include "net/routing/routing.h"
#include "random.h"
#include "net/netstack.h"
#include "net/ipv6/simple-udp.h"
#include <stdint.h>
#include <inttypes.h>
#include "sys/log.h"

#define LOG_MODULE "App"
#define LOG_LEVEL LOG_LEVEL_INFO

#define UDP_CLIENT_PORT    8765
#define UDP_SERVER_PORT    5678
#define SEND_INTERVAL      (10 * CLOCK_SECOND)

static struct simple_udp_connection udp_conn;
static uint32_t rx_count;

PROCESS(udp_client_process, "UDP client");
AUTOSTART_PROCESSES(&udp_client_process);

static void
udp_rx_callback(struct simple_udp_connection *c,
                const uip_ipaddr_t *sender_addr,
                uint16_t sender_port,
                const uip_ipaddr_t *receiver_addr,
                uint16_t receiver_port,
                const uint8_t *data,
                uint16_t datalen)
{
  LOG_INFO("Received response '%.*s' from ", datalen, (char *)data);
  LOG_INFO_6ADDR(sender_addr);
  LOG_INFO_("\n");
  rx_count++;
}

PROCESS_THREAD(udp_client_process, ev, data)
{
  static struct etimer periodic_timer;
  static char str[32];
  static uint32_t tx_count;
  static uint32_t missed_tx_count;
  uip_ipaddr_t dest_ipaddr;

  PROCESS_BEGIN();

  simple_udp_register(&udp_conn, UDP_CLIENT_PORT, NULL,
                      UDP_SERVER_PORT, udp_rx_callback);

  /* Desynchronize first TX so all motes do not collide */
  etimer_set(&periodic_timer, random_rand() % SEND_INTERVAL);

  while(1) {
    PROCESS_WAIT_EVENT_UNTIL(etimer_expired(&periodic_timer));

    if(NETSTACK_ROUTING.node_is_reachable() &&
       NETSTACK_ROUTING.get_root_ipaddr(&dest_ipaddr)) {

      if(tx_count % 10 == 0) {
        LOG_INFO("Tx/Rx/MissedTx: %" PRIu32 "/%" PRIu32 "/%" PRIu32 "\n",
                 tx_count, rx_count, missed_tx_count);
      }

      LOG_INFO("Sending request %" PRIu32 " to ", tx_count);
      LOG_INFO_6ADDR(&dest_ipaddr);
      LOG_INFO_("\n");

      snprintf(str, sizeof(str), "hello %" PRIu32, tx_count);
      simple_udp_sendto(&udp_conn, str, strlen(str), &dest_ipaddr);
      tx_count++;
    } else {
      LOG_INFO("Not reachable yet\n");
      if(tx_count > 0) {
        missed_tx_count++;
      }
    }

    /* ~10 s period plus ±1 s jitter */
    etimer_set(&periodic_timer,
               SEND_INTERVAL - CLOCK_SECOND + (random_rand() % (2 * CLOCK_SECOND)));
  }

  PROCESS_END();
}
```


### Makefile base

```make
CONTIKI_PROJECT = udp-client udp-server
all: $(CONTIKI_PROJECT)

CONTIKI = ../..
include $(CONTIKI)/Makefile.include
```


#### Actividades

Las actividades abarcan el uso y adaptación de las aplicaciones **UDP cliente**, **UDP servidor** y **temporizadores** (`etimer`) sobre una red RPL autónoma. Para interpretar la estructura programática de Contiki-NG considere el código previo.

| N° | Pregunta |
| - | - |
| 1 | Simule en Cooja una red RPL-Lite con **un** nodo `udp-server.c` (raíz del DODAG) y varios nodos `udp-client.c` que generen datos sensados arbitrarios en el *payload* UDP (por ejemplo `"id=%u seq=%lu t=%d"`). En el cliente, el envío periódico usa `etimer` y `SEND_INTERVAL`. Modifique el periodo a $10^{-5}\,\texttt{CLOCK\_SECOND}$, $10^{-10}\,\texttt{CLOCK\_SECOND}$ y $10^{-20}\,\texttt{CLOCK\_SECOND}$. Analice los tres casos (convergencia RPL, colisiones en el *Timeline*, mensajes `App`, y si el temporizador llega a expirar). Luego explique sus observaciones. |
| 2 | En este escenario **no** se usa `tunslip6`: el servidor UDP es a la vez raíz RPL y el cliente obtiene su dirección con `NETSTACK_ROUTING.node_is_reachable()` y `get_root_ipaddr()`. Explique cómo funciona `simple_udp_register` / `simple_udp_sendto` y el *callback* de recepción. Luego compare: (a) esta red RPL autónoma en Cooja; (b) un enrutador de borde con `tunslip6` (interfaz virtual `tun`); (c) un enrutador de borde nativo. ¿Cuándo haría falta `tunslip6` y cuándo basta el ejemplo `rpl-udp`? |
| 3 | Compare **RPL Lite** (por defecto) y **RPL Classic** (`MAKE_ROUTING = MAKE_ROUTING_RPL_CLASSIC`) sobre la misma topología UDP servidor/cliente. En cada escenario agregue nodos y sepárelos para forzar **múltiples saltos**. Desde la raíz (servidor) haga *ping* IPv6 a cada cliente (shell de Contiki-NG) y, en paralelo, mida el RTT de la petición/respuesta UDP (`hello N` / eco). ¿Qué valor de TTL observa en cada nodo? ¿Cuál es el tiempo de respuesta ICMP y el de la aplicación UDP? Compare Lite y Classic (almacenamiento *storing* vs *non-storing*, tamaño de paquetes descendentes, colisiones y `Tx/Rx/MissedTx`). |
| 4 | El ejemplo Contiki-OS `sky-websense.c` expone sensores por HTTP y se consulta con el navegador. En Contiki-NG ese patrón no se mantiene: el equivalente de estudio es `rpl-udp`. Indague `sky-websense.c` y proponga una integración hacia Contiki-NG: leer el sensor (o un valor arbitrario) en `udp-client.c`, enviarlo en el datagrama UDP y registrarlo en `udp-server.c`. Recopile los datos desde la salida `App` del servidor (filtro en *Mote output*). No se exige el navegador; si desea un visor web, justifique por qué iría en el *sink*/anfitrión y no en cada mote. |

# ACTIVIDADES RESUELTAS

### Actividad 1 - Temporizadores, convergencia y carga UDP

Se prepararon cuatro motes en cadena: raíz en x=0 y clientes en x=40, 80 y 120 m; el rango de transmisión es 50 m. El cliente envía una muestra determinista (`id`, secuencia y valor), y el servidor raíz la registra y hace eco. En Cooja, `CLOCK_SECOND=1000`: `10^-5 * CLOCK_SECOND` da 0.01 tick y se trunca a cero; `10^-10` y `10^-20` también quedan en cero. Por eso los tres casos extremos tienen el mismo temporizador efectivo y no representan esas tasas físicas de envío.

| Caso RPL-Lite | Tiempo del registro | `Tx/Rx/MissedTx` del cliente 2 | Observación |
|---|---:|---:|---|
| Control, 10 s | 113.985 s | 10/9/0 | Envío periódico; el último eco aún podía estar en vuelo al leerse el contador. |
| `10^-5 * CLOCK_SECOND` | 10.344792 s | 30/1/0 | Temporizador de cero ticks; tráfico inmediato tras ser alcanzable. |
| `10^-10 * CLOCK_SECOND` | 10.344792 s | 30/1/0 | Mismo resultado por cuantización a cero. |
| `10^-20 * CLOCK_SECOND` | 10.344792 s | 30/1/0 | Mismo resultado por cuantización a cero. |

En los tres casos extremos el sink registró la primera muestra en 10.3182 s, evidencia de que RPL ya permitía enviar. `Tx/Rx` muestra que pocos ecos llegaron durante la ráfaga. Es compatible con congestión o pérdidas, pero el modo headless no muestra el Timeline: no se puede afirmar que cada pérdida fuera causada por una colisión observada. Tampoco es fiable interpretar `RTT_ticks=2` en estos casos, porque los envíos solapados pisan la marca de tiempo del cliente.

### Actividad 2 - `simple_udp` y túnel IPv6

`simple_udp_register()` asocia una conexión UDP a un puerto local y registra el callback para los datagramas que coinciden con los filtros opcionales de origen. El callback recibe direcciones, puertos y payload con su longitud; debe tratarlo como bytes y no asumir que termina en NUL. `simple_udp_sendto()` envía los bytes indicados a una dirección IPv6 y al puerto remoto de la conexión. El eco no es automático: el callback del servidor lo envía explícitamente. RPL proporciona el encaminamiento, y `node_is_reachable()` junto a `get_root_ipaddr()` evitan transmitir antes de que el cliente tenga ruta a la raíz.

| Escenario | Función de `tunslip6` |
|---|---|
| RPL autónomo `rpl-udp` en Cooja | No hace falta: raíz y clientes se comunican dentro de la simulación. |
| Border router embebido + SLIP | Crea la interfaz `tun` del host y conecta IPv6 del host con la red de motes a través del enlace serie. |
| Border router nativo | Las capas IP corren en el host y un mote entrega la radio por SLIP; no equivale al servidor UDP autónomo. |

### Actividad 3 - RPL-Lite, RPL-Classic, saltos y RTT

Se compararon ambas variantes sobre la misma cadena de cuatro motes. Desde la raíz se hizo ping a los tres clientes, ubicados a uno, dos y tres saltos. El TTL/Hop Limit recibido y los RTT medidos fueron:

| Cliente | Saltos | Lite: Hop Limit / ICMP | Classic: Hop Limit / ICMP | Lite: RTT UDP medio | Classic: RTT UDP medio |
|---|---:|---:|---:|---:|---:|
| Mote 2 | 1 | 64 / 70 ms | 64 / 76 ms | 51.67 ms (9 ecos) | 45.78 ms (9 ecos) |
| Mote 3 | 2 | 63 / 95 ms | 63 / 119 ms | 93.12 ms (8 ecos) | 95.44 ms (9 ecos) |
| Mote 4 | 3 | 62 / 152 ms | 62 / 125 ms | 149.86 ms (7 ecos) | 165.89 ms (9 ecos) |

Lite usa por defecto modo descendente non-storing: la raíz mantiene información para construir source routes; el encabezado puede crecer con el número de saltos. Classic usa por defecto storing: los routers mantienen rutas descendentes distribuidas, a cambio de más estado en los nodos. En las corridas de control, el cliente 2 registró `Tx/Rx/MissedTx=10/9/0` en ambas variantes. Son resultados de una corrida por protocolo y topología, no una conclusión estadística general; los contadores no determinan por sí solos la causa de una colisión.

### Actividad 4 - Adaptación de `sky-websense.c`

El patrón propuesto para Contiki-NG es sensor o valor arbitrario -> cliente -> datagrama UDP -> sink. El cliente incluido genera una muestra reproducible y transmite identificador, secuencia y valor; el servidor registra emisor y payload en `App` y devuelve el datagrama. En una plataforma con sensor disponible, se sustituye la función de muestra por la lectura del sensor habilitado. Un visor HTTP, si se desea, conviene ubicarlo en el sink/host, donde hay más memoria y almacenamiento, en lugar de replicarlo en cada mote con recursos limitados.

**Ejecución comprobada:** los cinco escenarios Cooja terminaron con `TEST OK`; los logs y trazas están en `evidencia/`. No se generaron capturas gráficas de Timeline porque las corridas fueron headless. La lista de evidencia pendiente está al final de este notebook y en `Pendientes_Tarea3.md`.


# Solución — Tarea 3: uIP y RPL

**Asignatura:** TEL329 Redes de Sensores · 2026  
**Proyecto Contiki-NG:** `solucion_rpl_udp/`  
**Repositorio Contiki-NG:** `E:\contiki-ng` (en WSL: `/mnt/e/contiki-ng`)

## 1. Temporizadores y carga UDP en Cooja

El cliente de `rpl-udp` se modifica para enviar `id=<mote> seq=<n> t=<valor>` y mantener una raíz UDP que imprime cada muestra y devuelve el mismo datagrama. Los fuentes listos para compilar están en `solucion_rpl_udp/udp-client.c` y `solucion_rpl_udp/udp-server.c`. `PERIOD_CASE=0` es el control de 10 s; `1`, `2` y `3` representan los periodos solicitados. El valor sensado es una secuencia determinista arbitraria, adecuada para probar el formato y el recorrido extremo a extremo.

| Periodo pedido | Expresión en ticks | Resultado en Cooja (`CLOCK_SECOND=1000`) | Consecuencia esperable |
|---|---:|---:|---|
| `10^-5 * CLOCK_SECOND` | `CLOCK_SECOND / 100000` | 0 ticks (0,01 tick antes de truncar) | El `etimer` queda vencido inmediatamente; envío en cuanto el proceso vuelve a ejecutarse. No equivale a 100 000 envíos/s. |
| `10^-10 * CLOCK_SECOND` | `CLOCK_SECOND / 10000000000` | 0 ticks | Igual resolución efectiva que el caso anterior; más intentos de envío de los que la simulación puede procesar por tiempo lógico. |
| `10^-20 * CLOCK_SECOND` | 0 ticks | 0 | También se cuantiza a cero; no es físicamente distinguible de los dos casos anteriores usando `etimer`. |

`etimer` usa `clock_time_t`, un contador entero de ticks. Para los motes Cooja de esta versión, `CLOCK_CONF_SECOND=1000`; incluso `10^-5 * CLOCK_SECOND = 0,01` tick es inferior a la resolución y se trunca a cero. Los otros factores también producen cero. Una expresión como `10^-5` tampoco es notación de potencia en C (`^` es XOR); se debe escribir como una constante decimal/división y convertir a ticks. En los tres casos sub-tick se usa una rama de preprocesador para no compilar el `% SEND_INTERVAL` original: una comprobación `if` de ejecución no basta, porque GCC detecta que el divisor es constante cero y rechaza el código con `-Werror=div-by-zero`.

**Resultados medidos (Cooja, Lite, cuatro motes):** las tres compilaciones sub-tick producen exactamente el mismo `SEND_INTERVAL=0`. Tras alcanzar el root, el cliente 2 generó 30 solicitudes antes de que el test de Cooja registrara su estadística a `10,344792 s` de tiempo simulado: `Tx/Rx/MissedTx = 30/1/0`. El sink registró la primera muestra a `10,3182 s`; la prueba terminó correctamente con `TEST OK`. Esto confirma convergencia suficiente para enviar/recibir tráfico y la expiración inmediata del temporizador. La diferencia entre los exponentes no es observable a resolución de `etimer`. El único `RTT_ticks=2` capturado en cada corrida extrema **no es una medición fiable de ida y vuelta**: los envíos se solapan y la marca de tiempo única del cliente se sobrescribe. `MissedTx=0` solo indica que la ruta no se perdió; no significa que los ecos llegaran.

| Caso Lite | Tiempo simulado al registro | `Tx/Rx/MissedTx` del mote 2 | Resultado |
|---|---:|---:|---|
| Control `10 * CLOCK_SECOND` | 113,985 s | 10/9/0 | Tráfico periódico de 10 s; el contador se lee al décimo envío, con un eco aún pendiente. |
| `10^-5 * CLOCK_SECOND` | 10,345 s | 30/1/0 | Temporizador de cero ticks; ráfaga de solicitudes y pocos ecos observados antes de cerrar la prueba. |
| `10^-10 * CLOCK_SECOND` | 10,345 s | 30/1/0 | Igual que el anterior por cuantización a cero. |
| `10^-20 * CLOCK_SECOND` | 10,345 s | 30/1/0 | Igual que los otros factores sub-tick. |

La diferencia `Tx`/`Rx` muestra pérdidas de datagramas o ecos bajo la ráfaga, consistente con contención y colisiones en la topología. No es un contador de colisiones: el modo headless usado no expone el Timeline gráfico ni permite atribuir cada pérdida a una colisión. Los `.csc` generados conservan el plugin Timeline para inspección gráfica posterior.

### Registro de ejecución y evidencia

Se inició Docker Desktop. El motor respondió como Docker 29.0.1; la imagen `contiker/contiki-ng:latest`, el repositorio (`E:\contiki-ng`), el submódulo Cooja y la caché de Gradle quedaron en el almacenamiento Docker del disco E (`E:\DockerData\docker_data.vhdx`). Se compiló y ejecutó Cooja headless desde esa imagen; las cinco simulaciones terminaron con `TEST OK`.

Los `.csc` de `solucion_rpl_udp/cooja/` tienen cuatro motes en línea (root en x=0; clientes x=40, 80 y 120; rango de transmisión 50 y de interferencia 100). En Lite se ejecutaron los tres periodos extremos y el control de 10 s; en Classic se ejecutó el control sobre la misma topología. Para el control, el ScriptRunner envió los comandos `ping` por el shell serie del root y guardó RTT ICMP, Hop Limit, muestras UDP y RTT UDP en `evidencia/rpl-udp-case-*.txt`. Las salidas completas de Cooja/compilación están en los `.log` de esa carpeta. El modo headless no genera una captura de pantalla del Timeline; las pérdidas se describen como consistentes con contención, no como un recuento visual de colisiones.

Para repetir una corrida individual en Windows con Docker iniciado:

```powershell
docker run --rm `
  --mount "type=bind,source=E:\contiki-ng,destination=/home/user/contiki-ng" `
  --mount "type=bind,source=E:\redes de sensores,destination=/workspace/task" `
  --mount "type=volume,source=contiki-gradle-cache,destination=/home/user/.gradle" `
  contiker/contiki-ng:latest bash -lc 'cd /home/user/contiki-ng/examples/rpl-udp-tarea3 && make TARGET=cooja clean && cd /home/user/contiki-ng/tools/cooja && ./gradlew run --args="--no-gui /workspace/task/solucion_rpl_udp/cooja/rpl-udp-case-1-lite.csc"'
```

## 2. `simple_udp` y los tres escenarios de conectividad

`simple_udp_register(&conn, local_port, peer_addr, peer_port, callback)` enlaza el socket UDP simplificado al puerto local, configura el filtro opcional de dirección/puerto remoto y registra la función llamada al llegar un datagrama coincidente. Con `peer_addr == NULL`, se aceptan direcciones de origen arbitrarias (sujeto al puerto remoto configurado). El callback recibe la conexión, dirección y puerto de origen/destino, puntero al payload y longitud; debe tratar `data` como bytes con longitud explícita, no asumir que termina en `\0`. El servidor usa 5678 y escucha clientes desde 8765.

`simple_udp_sendto(&conn, data, len, &dest)` envía esos `len` bytes desde el puerto local asociado a `conn` a la dirección IPv6 `dest` y el puerto remoto con que se registró. No abre una ruta IP externa ni realiza el eco automáticamente: el callback del servidor invoca `sendto` explícitamente para devolver el payload al remitente. RPL proporciona la ruta; `node_is_reachable()`/`get_root_ipaddr()` evitan mandar antes de que el nodo tenga ruta y dirección de raíz.

| Arquitectura | Dónde está el root/BR | Qué conecta | Uso |
|---|---|---|---|
| RPL autónoma (`rpl-udp` en Cooja) | Un mote simulado hace `root_start()` y también es servidor UDP | Solo motes de la simulación | Basta para probar DODAG, tráfico y aplicación. No necesita `tunslip6` porque no hay que sacar paquetes a una red del anfitrión. |
| BR embebido + `tunslip6` | El mote físico/simulado ejecuta el border router; `tunslip6` corre en Linux y crea `tun` | Malla IPv6 ↔ anfitrión/red externa | Hace falta si el host debe intercambiar IPv6 con la red de motes mediante SLIP. `tun` es la interfaz virtual del host; serial socket/SLIP conecta con el mote. |
| BR nativo | El proceso BR corre en el anfitrión; un mote hace de `slip-radio` | Radio del mote ↔ pila IP en el host | Alternativa para mantener la capa de radio en el mote y ejecutar las capas superiores en el host; no es la topología de un servidor UDP autónomo en Cooja. |

Por tanto `tunslip6` es necesario para exponer/alcanzar la malla desde el host a través de la interfaz tun; no es necesario para el ejemplo `rpl-udp` ni para que clientes y root simulados se comuniquen dentro de Cooja.

## 3. RPL Lite frente a RPL Classic, multi-salto y RTT

Se ejecutó Lite —predeterminado en Contiki-NG— y luego Classic (`MAKE_ROUTING=MAKE_ROUTING_RPL_CLASSIC`), limpiando y recompilando entre variantes. La cadena de nodos está separada 40 m con rango de transmisión 50 m, por lo que los clientes 2, 3 y 4 quedan a uno, dos y tres enlaces del root. El servidor incluye `serial-shell` y el ScriptRunner ejecutó `ping` desde ese root a cada IPv6 de cliente.

| Propiedad | RPL Lite (predeterminado) | RPL Classic |
|---|---|---|
| Modo descendente por defecto | Non-storing | Storing (sin multicast); puede cambiarse con `RPL_CONF_MOP` |
| Estado de rutas hacia abajo | La raíz conserva enlaces/DAO y arma source-routing header para llegar al destino | Los routers intermedios mantienen entradas de ruta descendente distribuidas mediante DAO |
| Memoria | Menos tabla descendente en nodos intermedios; más responsabilidad de la raíz | Más estado de rutas por nodo, que crece con destinos/subárboles |
| Paquete descendente | El source-routing header crece con la longitud del camino y puede fragmentarse o superar MTU | Normalmente el datagrama se reenvía salto a salto sin toda la lista de ruta en cada paquete; añade control/estado de rutas |
| Sensibilidad a tráfico | Source route puede aumentar tamaño/tiempo de aire en caminos largos | DAO y mantenimiento de tablas consumen tráfico/ram; forwarding requiere entradas instaladas |

### Ping desde el root y RTT de aplicación

El shell de Contiki-NG muestra el Hop Limit recibido bajo la etiqueta `ttl`. Con un root directamente vecino, el destino de un enlace respondió con 64; los destinos a dos y tres enlaces respondieron con 63 y 62. En esta topología, los routers intermedios descuentan el límite en los paquetes de respuesta.

| Cliente | Saltos | Lite: Hop Limit / ICMP RTT | Classic: Hop Limit / ICMP RTT | Lite: RTT UDP medio | Classic: RTT UDP medio |
|---|---:|---:|---:|---:|---:|
| Mote 2 | 1 | 64 / 70 ms | 64 / 76 ms | 51,67 ms (9 ecos) | 45,78 ms (9 ecos) |
| Mote 3 | 2 | 63 / 95 ms | 63 / 119 ms | 93,12 ms (8 ecos) | 95,44 ms (9 ecos) |
| Mote 4 | 3 | 62 / 152 ms | 62 / 125 ms | 149,86 ms (7 ecos) | 165,89 ms (9 ecos) |

`CLOCK_SECOND=1000` en Cooja, así que cada `RTT_ticks` equivale a 1 ms. El RTT UDP incluye ida, procesamiento/eco en el root y vuelta; las medias se calcularon de los ecos registrados en las corridas de control. En esas corridas, el cliente 2 registró `Tx/Rx/MissedTx=10/9/0` al décimo envío (el último eco aún podía estar en vuelo). Los RTT cambian con los eventos discretos y la ruta. Una sola ejecución por protocolo/topología no demuestra una ventaja general de Lite o Classic. En los casos de temporizador cero las marcas UDP se pisan, así que sus RTT no se comparan.

## 4. De `sky-websense.c` a Contiki-NG

El patrón portable a Contiki-NG es sensor → aplicación del cliente → datagrama UDP → sink/servidor. La lectura real se hace con la API del sensor habilitado para la plataforma (`SENSORS_SENSOR(...).value(...)` tras activarlo); para una topología Cooja genérica el ejemplo incluido produce un dato arbitrario reproducible. El formato transmite identificador del mote, secuencia y muestra. El servidor registra emisor, payload y contador en `App` y responde por eco. Se puede filtrar `App` en Mote output para recolectar las muestras; no se precisa un navegador.

Si se añade un visor web, debe residir en el sink/anfitrión: allí se agregan los registros y hay más RAM/almacenamiento y capacidad de HTTP. Un servidor HTTP por mote duplica código/estado en dispositivos constreñidos, aumenta tráfico/consumo y no aporta valor al envío de mediciones. Para integrar un host externo, entonces sí corresponde exponer la malla mediante un BR y, en el escenario SLIP, `tunslip6`.

## Referencias

- Contiki-NG, [RPL tutorial](https://docs.contiki-ng.org/en/develop/doc/tutorials/RPL.html).
- Contiki-NG, [RPL border router](https://docs.contiki-ng.org/en/develop/doc/tutorials/RPL-border-router.html).
- Código base `examples/rpl-udp` de Contiki-NG (checkout en `E:\contiki-ng`).
- Notebook original `TEL329-Tarea3-2026-v1.0.ipynb`.


# Pendientes para dejar lista la entrega — Tarea 3

## Ya completado

- El notebook `TEL329-Tarea3-2026-v1.0.ipynb` contiene las respuestas a las cuatro actividades, conserva el material original y es JSON válido.
- Se compilaron y ejecutaron cinco escenarios Cooja: RPL Lite con control de 10 s y con los tres intervalos sub-tick; RPL Classic con el control de 10 s. Los cinco terminaron con `TEST OK`.
- Las mediciones de UDP, ping IPv6, Hop Limit y RTT están en `evidencia/rpl-udp-case-*.txt`; los logs de compilación/ejecución están en los `.log` correspondientes.
- El ejemplo usa una muestra sensada arbitraria y determinista, tal como permite la actividad; no depende de un sensor físico.

## Evidencia que todavía podría agregarse

### 1. Capturas gráficas de Cooja

Las pruebas se ejecutaron en modo headless. Por eso no hay capturas de pantalla de la topología, Timeline, Mote Output ni Radio Messages. Si la entrega exige evidencia visual, abrir los escenarios `.csc` de `solucion_rpl_udp/cooja/` en Cooja con interfaz gráfica y guardar las capturas en `evidencia/`. Para respaldar el análisis de colisiones, la captura debería mostrar las transmisiones superpuestas en Timeline o Radio Messages. Los contadores `Tx/Rx/MissedTx` por sí solos no identifican la causa de cada pérdida.

### 2. Túnel IPv6 con `tunslip6` — solo si se exige ejecutar los pasos introductorios

No se ejecutó un border router conectado a `tun0`: las simulaciones medidas usan el ejemplo RPL-UDP autónomo, donde el servidor es la raíz y no hace falta `tunslip6`. Si también deben demostrarse los pasos 1 y 2 del notebook, falta correr el escenario separado de border router, verificar `ifconfig tun0` y hacer `ping6` al router; guardar sus salidas o capturas en `evidencia/`.

## Alcance de la comparación

Los RTT y Hop Limit de Lite y Classic corresponden a una corrida de control por protocolo en la misma topología de cuatro motes. Sirven como resultados observados de estos ensayos, pero no como comparación estadística general. Repetir con varias semillas y recopilar los contadores de todos los clientes sería una ampliación opcional si se pide mayor rigor.

## Ubicación

Notebook, informe, escenarios y evidencia están en `E:\redes de sensores`. La única limitación actual de evidencia es la falta de capturas gráficas y, si la pauta lo requiere por separado, la demostración del túnel `tunslip6`.
